# Fig. 2 Scaling Diagnostic: Does the transition scale with model size?

Nick's question was roughly:

> Can we estimate a constant or exponent like \(N^x / \#\mathrm{params}\) at the point where generalization becomes larger than 0.5?

This notebook treats that as an **exploratory diagnostic**, not a poster headline. The current experiment only has three architecture points (`UNet-64`, `UNet-128`, `UNet-256`), and the crossing depends on the feature space and threshold definition. The goal here is to make the calculation explicit and decide whether the idea is worth pursuing later.

## Definition

For each architecture, define

\[
N_{50} = \min N_{2D}\ \mathrm{where}\ \mathrm{GL}(N_{2D}) \ge 0.5,
\]

with interpolation in \(\log N_{2D}\) between sampled training-set sizes.

Here \(\mathrm{GL}\) is the generalization score used in the Fig. 2-style analysis: the fraction of generated maps that are not closer to the training set than the real-data calibrated threshold.

Then fit the rough relation

\[
N_{50} = C P^\alpha,
\]

where \(P\) is the number of trainable UNet parameters. Equivalently,

\[
P = K N_{50}^{1/\alpha}.
\]

This is only meaningful if the crossings are reliable and monotonic.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

TABLE_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2' / 'tables'
SCRIPT = PROJECT_DIR / 'scripts' / 'estimate_nf_generalize_scaling.py'

PCA_TABLE = TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_metrics.csv'
SSCD_TABLE = TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_metrics.csv'

print('PROJECT_DIR =', PROJECT_DIR)
print('PCA table exists:', PCA_TABLE.exists(), PCA_TABLE)
print('SSCD table exists:', SSCD_TABLE.exists(), SSCD_TABLE)
print('scaling script exists:', SCRIPT.exists(), SCRIPT)

## Run the scaling script

The script writes:

- `nf_generalize_fig2_n50_scaling_summary.csv`
- `nf_generalize_fig2_n50_scaling_fits.json`

Run this once per threshold choice. Start with the same q95-calibrated threshold used in the poster, then compare q99 or fixed thresholds as a sensitivity check.

In [ ]:
def run_scaling(score_mode='adaptive', quantile='q95', tau=0.9, threshold=0.5):
    if not SCRIPT.exists():
        raise FileNotFoundError(SCRIPT)
    if not (PCA_TABLE.exists() or SSCD_TABLE.exists()):
        display(Markdown(
            '**Metric tables are missing locally.** Sync/run the Fig. 2 PCA and SSCD analyzers first.\n\n'
            'On Great Lakes, expected files are under `results/nf_generalize_fig2/tables/`.'
        ))
        return None

    cmd = [
        sys.executable,
        str(SCRIPT),
        '--project-dir', str(PROJECT_DIR),
        '--score-mode', score_mode,
        '--quantile', quantile,
        '--tau', str(tau),
        '--threshold', str(threshold),
    ]
    print(' '.join(cmd))
    result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'scaling script failed with code {result.returncode}')
    return result

# Main poster-consistent diagnostic.
run_scaling(score_mode='adaptive', quantile='q95', threshold=0.5)

In [ ]:
SUMMARY_PATH = TABLE_DIR / 'nf_generalize_fig2_n50_scaling_summary.csv'
FITS_PATH = TABLE_DIR / 'nf_generalize_fig2_n50_scaling_fits.json'

if SUMMARY_PATH.exists():
    summary = pd.read_csv(SUMMARY_PATH)
    display(summary)
else:
    summary = pd.DataFrame()
    display(Markdown('No scaling summary found yet.'))

if FITS_PATH.exists():
    fits = json.loads(FITS_PATH.read_text())
    display(fits)
else:
    fits = []

## Plot \(N_{50}\) against model parameters

This is the core sanity check. If the scaling idea were clean, the points would lie roughly on a line in log-log space for both PCA and SSCD. If PCA and SSCD disagree strongly, or one architecture is censored/nonmonotonic, then the exponent should not be treated as physical.

In [ ]:
ARCH_LABELS = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256'}
FEATURE_COLORS = {'PCA': '#0072B2', 'SSCD': '#D55E00'}
MARKERS = {'u64': '^', 'u128': 'o', 'u256': 's'}

required_cols = {'feature', 'arch', 'model_params', 'n50_2d_images', 'n50_status'}
if summary.empty:
    display(Markdown('No summary table loaded; skipping plot.'))
elif not required_cols.issubset(summary.columns):
    missing = sorted(required_cols - set(summary.columns))
    display(Markdown(f'Scaling summary is missing required columns: `{missing}`. Re-run the scaling script.'))
else:
    plot_df = summary.copy()
    plot_df['model_params'] = pd.to_numeric(plot_df['model_params'], errors='coerce')
    plot_df['n50_2d_images'] = pd.to_numeric(plot_df['n50_2d_images'], errors='coerce')
    valid = plot_df[
        np.isfinite(plot_df['model_params'])
        & np.isfinite(plot_df['n50_2d_images'])
        & (plot_df['model_params'] > 0)
        & (plot_df['n50_2d_images'] > 0)
    ].copy()

    if valid.empty:
        display(Markdown(
            '**No positive finite rows are available for the log-scale N50 plot.**\n\n'
            'This usually means the metric tables were missing, the scaling script did not infer model parameters, '
            'or all crossings are unavailable/censored. Inspect the summary below before interpreting scaling.'
        ))
        display(summary)
    else:
        fig, ax = plt.subplots(figsize=(7.4, 5.4), dpi=140, constrained_layout=True)
        for feature, sub in valid.sort_values('model_params').groupby('feature'):
            color = FEATURE_COLORS.get(feature, '0.3')
            ax.plot(
                sub['model_params'], sub['n50_2d_images'],
                color=color, lw=2.5, alpha=0.75,
                label=f'{feature} N50'
            )
            for _, row in sub.iterrows():
                arch = str(row['arch'])
                status = str(row['n50_status'])
                ax.scatter(
                    row['model_params'], row['n50_2d_images'],
                    s=110, marker=MARKERS.get(arch, 'o'),
                    color=color,
                    edgecolor='white', linewidth=1.0, zorder=3,
                )
                ax.annotate(
                    f"{ARCH_LABELS.get(arch, arch)}\n{status}",
                    (row['model_params'], row['n50_2d_images']),
                    xytext=(6, 5), textcoords='offset points', fontsize=9,
                )

        ax.set_xscale('log')
        ax.set_yscale('log', base=2)
        ax.set_xlabel('Trainable UNet parameters')
        ax.set_ylabel(r'$N_{50}$ training images')
        ax.set_title(r'Data size where generalization first exceeds 0.5')
        ax.grid(True, which='both', alpha=0.22)
        ax.legend(frameon=False)
        out = TABLE_DIR / 'nf_generalize_fig2_n50_scaling_plot.png'
        fig.savefig(out, bbox_inches='tight', dpi=240)
        plt.show()
        print('wrote', out)

        dropped = plot_df.loc[~plot_df.index.isin(valid.index)]
        if len(dropped):
            display(Markdown('Rows dropped from the log plot because `model_params` or `n50_2d_images` was missing/non-positive:'))
            display(dropped)

## Sensitivity check

The exponent is only useful if it does not move wildly when the feature space or threshold changes. Run these variants and compare:

- q95 adaptive threshold: poster-style baseline.
- q99 adaptive threshold: stricter notion of being far from training.
- fixed threshold, e.g. \(\tau = 0.9\): useful only if that column exists in the metrics table.

If the crossing rank/order changes substantially, the safe conclusion is: **the transition exists, but the current data are not enough to estimate a universal scaling exponent.**

In [ ]:
# Optional sensitivity runs. Uncomment on Great Lakes once the metric tables are present.
# run_scaling(score_mode='adaptive', quantile='q99', threshold=0.5)
# run_scaling(score_mode='fixed', tau=0.9, threshold=0.5)

## Interpretation template

Use this language if the fit looks reasonable:

> As a first diagnostic, I define \(N_{50}\) as the number of 2D training maps where the generalization score crosses 0.5. Fitting \(N_{50}\sim P^\alpha\) across UNet-64/128/256 gives a rough scaling, but with only three architectures and feature-dependent thresholds this is not yet a law.

Use this language if PCA and SSCD disagree or the fit is unstable:

> The transition is robust as a qualitative phenomenon, but the exponent estimate is not robust. It depends on the feature space and threshold choice, so I would treat model-size scaling as a follow-up experiment requiring more architectures and a fixed validation protocol.

This is probably the current best scientific stance for the poster/workshop.